# 04 — Model Training

**IBM Bob assisted** — All training code generated via IBM Bob Phase 4.

Trains: Logistic Regression · Random Forest (GridSearchCV) · XGBoost with early stopping

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath('..'))
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from dashboard.components.model_trainer import load_features, train_all, MODELS_DIR

plt.rcParams['figure.dpi'] = 120
print('Imports OK')

In [ ]:
X, y, feature_names = load_features()
print(f'Feature matrix: X={X.shape}  y={y.shape}')
print(f'Class balance: GO={y.sum()}  SCRUB={(y==0).sum()}')

In [ ]:
results = train_all(X, y, feature_names)

In [ ]:
# Performance table
metrics_df = pd.DataFrame(results['metrics'])
display(metrics_df.set_index('model').T)

In [ ]:
# Bar chart: ROC-AUC comparison
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#0f62fe','#ee538b','#42be65']
for i, row in metrics_df.iterrows():
    ax.bar(row['model'], row['roc_auc'], color=colors[i % len(colors)])
ax.set_ylim(0, 1.05)
ax.axhline(0.5, color='gray', lw=0.8, ls='--', label='Random baseline')
ax.set_title('Model ROC-AUC Comparison', fontsize=13)
ax.set_ylabel('ROC-AUC')
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/model_comparison_auc.png', bbox_inches='tight')
plt.show()

In [ ]:
# Confusion matrix for best model
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
best = results['best_model']
X_test = results['X_test']
y_test = results['y_test']
y_pred = best.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5,4))
disp = ConfusionMatrixDisplay(cm, display_labels=['SCRUB','GO'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion Matrix — {results["best_model_name"]}', fontsize=12)
plt.tight_layout()
plt.savefig('../data/processed/confusion_matrix_best.png', bbox_inches='tight')
plt.show()

In [ ]:
# Random Forest feature importance
if hasattr(results['models']['random_forest'], 'feature_importances_'):
    importances = results['models']['random_forest'].feature_importances_
    idx = np.argsort(importances)[::-1]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(range(len(importances)), importances[idx], color='#0f62fe')
    ax.set_xticks(range(len(importances)))
    ax.set_xticklabels([feature_names[i] for i in idx], rotation=45, ha='right')
    ax.set_title('Random Forest Feature Importances', fontsize=13)
    ax.set_ylabel('Importance')
    plt.tight_layout()
    plt.savefig('../data/processed/rf_feature_importance.png', bbox_inches='tight')
    plt.show()